# How much of each outlet's WNBA coverage touches race, protest, or LGBTQ topics?

The patterns below are the exact ones the piece's charts use.
An article counts once if it matches anything in a pattern.

In [ ]:
import re
import pandas as pd

RACE_PROTEST = re.compile(
    r"racis|white privilege|racial|\b(?:black|white) (?:wom[ae]n|player|players|athlete|athletes|star|stars|community)\b|"
    r"social justice|black lives matter|\bblm\b|\banthem\b|protest|kneel|boycott|police shooting|shooting of|\belection\b|indictment",
    re.I)
LGBTQ = re.compile(
    r"lgbtq?\b|lesbian|\bgay\b|\bqueer\b|sexual orientation|openly gay|same-sex|"
    r"athlete ally|pride night|pride month|bisexual|non-?binary", re.I)

In [ ]:
OUTLETS = {
    "Fox News": "../data/fox_wnba.csv",
    "Just Women's Sports": "../data/just_womens_sports.csv",
    "ESPN": "../data/espn.csv",
}

def yearly_rates(path, pattern):
    df = pd.read_csv(path)
    text_col = "title" if "title" in df else "headline"
    text = df[text_col].fillna("") + " " + df.get("body", "").fillna("")
    df = df.assign(year=df["date"].str[:4], hit=text.str.contains(pattern))
    df = df[df["year"] >= "2019"]
    out = df.groupby("year").agg(articles=("hit", "size"), matching=("hit", "sum"))
    out["share"] = (out.matching / out.articles * 100).round(1)
    return out[out.articles >= 10]

for name, path in OUTLETS.items():
    print(f"\n=== {name}: % touching race or protest ===")
    print(yearly_rates(path, RACE_PROTEST).to_string())

In [ ]:
for name, path in OUTLETS.items():
    print(f"\n=== {name}: % touching LGBTQ topics ===")
    print(yearly_rates(path, LGBTQ).to_string())